In [0]:
%run ./01-config

In [0]:
# ==============================================================================
# LOAD SET 2 - Copy test data files into the landing zone
# Only copies files that don't already exist in the destination.
# Runs only if Set 2 data is available, else runs Set 1 data.
# ==============================================================================

print("Loading data into landing zone...")

# Define set 2 and set 1 file mappings: (source_file, destination_folder)
set1_files = [
    (f"{test_data_dir}/1-registered_users_1.csv", f"{landing_zone}/registered_users_bz/"),
    (f"{test_data_dir}/5-gym_logins_1.csv", f"{landing_zone}/gym_logins_bz/"),
    (f"{test_data_dir}/2-user_info_1.json", f"{landing_zone}/kafka_multiplex_bz/"),
    (f"{test_data_dir}/3-bpm_1.json", f"{landing_zone}/kafka_multiplex_bz/"),
    (f"{test_data_dir}/4-workout_1.json", f"{landing_zone}/kafka_multiplex_bz/"),
]

set2_files = [
    (f"{test_data_dir}/1-registered_users_2.csv", f"{landing_zone}/registered_users_bz/"),
    (f"{test_data_dir}/5-gym_logins_2.csv", f"{landing_zone}/gym_logins_bz/"),
    (f"{test_data_dir}/2-user_info_2.json", f"{landing_zone}/kafka_multiplex_bz/"),
    (f"{test_data_dir}/3-bpm_2.json", f"{landing_zone}/kafka_multiplex_bz/"),
    (f"{test_data_dir}/4-workout_2.json", f"{landing_zone}/kafka_multiplex_bz/"),
]


# Check if all Set 2 files exist
try:
    for src, _ in set2_files:
        dbutils.fs.ls(src)
    set2_available = True
except Exception:
    set2_available = False
files_to_copy = set2_files if set2_available else set1_files
set_name = "Set 2" if set2_available else "Set 1"

print(f"Detected {set_name} data. Copying files...")

for src, dest in files_to_copy:
    file_name = src.split("/")[-1]
    dest_path = dest + file_name

    # Check if file already exists in destination
    try:
        dbutils.fs.ls(dest_path)
        print(f"  Skipped (already exists): {file_name}")
    except Exception:
        dbutils.fs.cp(src, dest_path)
        print(f"  Copied: {file_name} -> {dest}")

print(f"{set_name} data loading complete.")

In [0]:
import time
from pyspark.sql import functions as F

spark.sql(f"USE {catalog}.{db_name}")

start = int(time.time())
print("Starting bronze layer batch ingestion (idempotent)...")

# ==============================================================================
# BRONZE LAYER - BATCH INGESTION (IDEMPOTENT)
# Reads raw files from cloud storage and appends to Delta tables with metadata.
# Uses source_file tracking to ensure each file is only loaded once.
# ==============================================================================

# --- 1. Registered Users (CSV) ---
print("Ingesting registered_users_bz...", end='')
schema_users = "user_id long, device_id long, mac_address string, registration_timestamp double"
target_table = f"{catalog}.{db_name}.registered_users_bz"

loaded_files = set()
if spark.catalog.tableExists(target_table):
    loaded_files = {row.source_file for row in
                    spark.table(target_table).select("source_file").distinct().collect()}

df_registered_users = (spark.read
    .format("csv")
    .schema(schema_users)
    .option("header", True)
    .load(landing_zone + "/registered_users_bz")
    .withColumn("load_time", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)

if loaded_files:
    df_registered_users = df_registered_users.filter(~F.col("source_file").isin(loaded_files))

if df_registered_users.head(1):
    df_registered_users.write.mode("append").saveAsTable(target_table)
    print("Done (new files ingested)")
else:
    print("Skipped (all files already loaded)")

# --- 2. Gym Logins (CSV) ---
print("Ingesting gym_logins_bz...", end='')
schema_gym = "mac_address string, gym bigint, login double, logout double"
target_table = f"{catalog}.{db_name}.gym_logins_bz"

loaded_files = set()
if spark.catalog.tableExists(target_table):
    loaded_files = {row.source_file for row in
                    spark.table(target_table).select("source_file").distinct().collect()}

df_gym_logins = (spark.read
    .format("csv")
    .schema(schema_gym)
    .option("header", True)
    .load(landing_zone + "/gym_logins_bz")
    .withColumn("load_time", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)

if loaded_files:
    df_gym_logins = df_gym_logins.filter(~F.col("source_file").isin(loaded_files))

if df_gym_logins.head(1):
    df_gym_logins.write.mode("append").saveAsTable(target_table)
    print("Done (new files ingested)")
else:
    print("Skipped (all files already loaded)")

# --- 3. Kafka Multiplex (JSON) ---
print("Ingesting kafka_multiplex_bz...", end='')
schema_kafka = "key string, value string, topic string, partition bigint, offset bigint, timestamp bigint"
target_table = f"{catalog}.{db_name}.kafka_multiplex_bz"

loaded_files = set()
if spark.catalog.tableExists(target_table):
    loaded_files = {row.source_file for row in
                    spark.table(target_table).select("source_file").distinct().collect()}

# Load date dimension for enrichment
df_date_lookup = spark.table(f"{catalog}.{db_name}.date_lookup").select("date", "week_part")

df_kafka = (spark.read
    .format("json")
    .schema(schema_kafka)
    .load(landing_zone + "/kafka_multiplex_bz")
    .withColumn("load_time", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
    # Join with date_lookup to add date and week_part for partitioning
    .join(F.broadcast(df_date_lookup),
          [F.to_date((F.col("timestamp") / 1000).cast("timestamp")) == F.col("date")],
          "left")
)

if loaded_files:
    df_kafka = df_kafka.filter(~F.col("source_file").isin(loaded_files))

if df_kafka.head(1):
    df_kafka.write.mode("append").saveAsTable(target_table)
    print("Done (new files ingested)")
else:
    print("Skipped (all files already loaded)")

print(f"\nBronze layer batch ingestion completed in {int(time.time()) - start} seconds")

In [0]:
import time
sets = 1
start = int(time.time())
print(f"\nValidating bronze layer records...")

# Validate registered_users_bz
table_name = "registered_users_bz"
expected_count = 5 if sets == 1 else 10
actual_count = spark.read.table(f"{catalog}.{db_name}.{table_name}").count()
assert actual_count == expected_count, f"Expected {expected_count:,} records, found {actual_count:,} in {table_name}"
print(f"Found {actual_count:,} / Expected {expected_count:,} records: Success")

# Validate gym_logins_bz
table_name = "gym_logins_bz"
expected_count = 8 if sets == 1 else 16
actual_count = spark.read.table(f"{catalog}.{db_name}.{table_name}").count()
assert actual_count == expected_count, f"Expected {expected_count:,} records, found {actual_count:,} in {table_name}"
print(f"Found {actual_count:,} / Expected {expected_count:,} records: Success")

# Validate kafka_multiplex_bz by topic
table_name = "kafka_multiplex_bz"
actual_count = spark.read.table(f"{catalog}.{db_name}.{table_name}").where("topic='user_info'").count()
expected_count = 7 if sets == 1 else 13
assert actual_count == expected_count, f"Expected {expected_count:,} records, found {actual_count:,} in {table_name} where topic='user_info'"
print(f"Found {actual_count:,} / Expected {expected_count:,} records where topic='user_info': Success")

actual_count = spark.read.table(f"{catalog}.{db_name}.{table_name}").where("topic='workout'").count()
expected_count = 16 if sets == 1 else 32
assert actual_count == expected_count, f"Expected {expected_count:,} records, found {actual_count:,} in {table_name} where topic='workout'"
print(f"Found {actual_count:,} / Expected {expected_count:,} records where topic='workout': Success")

actual_count = spark.read.table(f"{catalog}.{db_name}.{table_name}").where("topic='bpm'").count()
expected_count = sets * 253801
assert actual_count == expected_count, f"Expected {expected_count:,} records, found {actual_count:,} in {table_name} where topic='bpm'"
print(f"Found {actual_count:,} / Expected {expected_count:,} records where topic='bpm': Success")

print(f"Bronze layer validation completed in {int(time.time()) - start} seconds")